In [1]:
!pip install -U datasets huggingface_hub pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's depe

| Library           | Single-line use                                                                              |
| ----------------- | -------------------------------------------------------------------------------------------- |
| `datasets`        | Used to load, create, preprocess, and manage datasets for training or fine-tuning models.    |
| `huggingface_hub` | Used to download/upload models, tokenizers, datasets, and checkpoints from Hugging Face Hub. |
| `pillow`          | Used to open, process, resize, and manipulate images in Python.                              |


In [1]:
import os
from datasets import Dataset, Features, Value, Image
# from huggingface_hub import login

| Code        | Single-line description                                                                |
| ----------- | -------------------------------------------------------------------------------------- |
| `import os` | Used to work with file paths, folders, and environment variables.                      |
| `Dataset`   | Used to create a Hugging Face dataset from Python data.                                |
| `Features`  | Used to define the schema/structure of dataset columns.                                |
| `Value`     | Used to define text, number, or string column types in the dataset.                    |
| `Image`     | Used to define an image column in the dataset.                                         |
| `login`     | Used to authenticate with Hugging Face Hub for uploading or accessing models/datasets. |


In [3]:
# 2) Your image folder
IMG_DIR = "./iphone_imgs"  # change this

In [4]:
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

In [5]:
image_files

['./iphone_imgs/image01.jpg',
 './iphone_imgs/image02.jpg',
 './iphone_imgs/image03.jpg',
 './iphone_imgs/image04.jpg',
 './iphone_imgs/image05.jpg']

In [6]:
# 3) Captions for images
captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

In [7]:
# assert len(image_files) == len(captions), "Images and captions count must match!"

In [8]:
instruction = "Describe this iPhone product image in one sentence."

In [9]:
rows = []

In [10]:
for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [11]:
rows

[{'image': './iphone_imgs/image01.jpg',
  'text': 'A white iPhone shown from the back and front on a white background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image01.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A white iPhone shown from the back and front on a white background.'}]}]},
 {'image': './iphone_imgs/image02.jpg',
  'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image02.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.'}]}]},
 {'image': './iphone_imgs/imag

In [ ]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

In [ ]:
ds = Dataset.from_list(rows, features=features)

| Column                        | Use                                                                             |
| ----------------------------- | ------------------------------------------------------------------------------- |
| `"image": Image()`            | Creates an image column where image files/paths are stored as image data.       |
| `"text": Value("string")`     | Creates a text column for captions, labels, or descriptions.                    |
| `"messages": Value("string")` | Creates a string column to store chat/instruction data |


In [13]:
# If you want to store messages as real objects, easiest is to NOT force Features for messages.
ds = Dataset.from_list(rows)  # messages stored as nested objects

In [14]:
# 4) Make train/test splits (with only 5 images, keep test=1)
splits = ds.train_test_split(test_size=1, seed=3407)

In [ ]:
# 5) Push to Hub
REPO_ID = "sunny199/iphone5_vlm_img"  # change this

In [ ]:
splits["train"].push_to_hub(REPO_ID, split="train")

In [ ]:
splits["test"].push_to_hub(REPO_ID, split="test")

In [ ]:
print("Uploaded:", REPO_ID)

In [15]:
from datasets import load_dataset

In [16]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [17]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [18]:
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")

In [19]:
print(train_ds)

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})


In [20]:
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7C577FAA1460>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K

Actual images COCO train2017 se aati hain. LLaVA GitHub clearly bolta hai ki visual instruction tuning ke liye annotation download karo, aur images constituting datasets se download karo; COCO ke liye train2017 images use hote hain.



https://huggingface.co/datasets/HuggingFaceM4/ChartQA

https://huggingface.co/datasets/nlphuji/flickr30k

https://huggingface.co/datasets/unsloth/LaTeX_OCR

https://huggingface.co/datasets/sunny199/iphone5_vlm_img

https://huggingface.co/datasets/sunny199/iphone5_vlm

| Value | Meaning                    |
| ----- | -------------------------- |
| `0`   | Human-generated question   |
| `1`   | Machine-generated question |


In [21]:
# # ================================
# # Qwen 2.5 VL
# # ================================
# model_name = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-32B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2 VL
# # ================================
# model_name = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2.5 Omni (Multimodal)
# # ================================
# model_name = "unsloth/Qwen2.5-Omni-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-Omni-7B-Instruct-bnb-4bit"

# # ================================
# # Llama Vision
# # ================================
# model_name = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit"

# # ================================
# # Gemma Vision
# # ================================
# model_name = "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/MedGemma-27B-Vision-Instruct-bnb-4bit"

# # ================================
# # Mistral Vision
# # ================================
# model_name = "unsloth/Pixtral-12B-2409-bnb-4bit"

# # ================================
# # LLaVA
# # ================================
# model_name = "unsloth/llava-1.5-7b-hf-bnb-4bit"
# model_name = "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit"

In [ ]:
"qwen2_vl_2b"
"qwen25_vl_3b"
"qwen25_omni_3b"
"medgemma_4b_vision"
"qwen25_vl_7b"
"qwen2_vl_7b"
"qwen25_omni_7b"
"llava15_7b"
"llava16_mistral_7b"

In [1]:
# --- Install (Colab) ---
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

# ==========================================================
# Universal Unsloth Vision Fine-tuning (Model + Dataset Dynamic)
# Supports: Qwen VL, LLaVA, Pixtral, MedGemma, Llama Vision
# Dataset selectable
# Trains on 150 row subset
# Runs for 2 Epochs
# ==========================================================

import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# ==========================================================
# 1️Model Registry
# ==========================================================
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

In [4]:
# ==========================================================
# Dataset Registry (Vision-ready datasets)
# ==========================================================
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "sunny199/iphone5_vlm",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [5]:
# ==========================================================
# Config
# ==========================================================
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "iphone_custom"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

| Parameter                              | One-line description                                                    |
| -------------------------------------- | ----------------------------------------------------------------------- |
| `model_key: str = "qwen2_vl_2b"`       | Defines which vision-language model will be used for fine-tuning.       |
| `dataset_key: str = "latex_ocr"`       | Defines which dataset will be used for training.                        |
| `subset_rows: int = 150`               | Uses only 150 rows from the dataset for quick demo/testing.             |
| `eval_ratio: float = 0.1`              | Keeps 10% of the data for evaluation/validation.                        |
| `seed: int = 3407`                     | Fixes randomness so results are reproducible.                           |
| `r: int = 16`                          | LoRA rank; controls how many trainable low-rank parameters are added.   |
| `lora_alpha: int = 16`                 | LoRA scaling factor; controls the strength of LoRA updates.             |
| `lora_dropout: float = 0.0`            | Dropout applied inside LoRA layers to reduce overfitting.               |
| `per_device_train_batch_size: int = 2` | Number of samples processed per GPU/device in one training step.        |
| `gradient_accumulation_steps: int = 4` | Accumulates gradients for 4 steps before updating weights.              |
| `num_train_epochs: int = 2`            | Trains the model for 2 full passes over the training dataset.           |
| `learning_rate: float = 2e-4`          | Controls how fast the model weights are updated during training.        |
| `logging_steps: int = 10`              | Logs training metrics after every 10 steps.                             |
| `weight_decay: float = 0.001`          | Regularization value used to reduce overfitting.                        |
| `max_length: int = 2048`               | Maximum token length allowed for model input/output sequence.           |
| `output_dir: str = "outputs"`          | Folder where training outputs/checkpoints can be saved.                 |
| `save_dir: str = "vlm_lora_output"`    | Final folder where the trained LoRA adapter/model output will be saved. |


In [6]:
# from datasets import load_dataset
# dataset = load_dataset("HuggingFaceM4/ChartQA")
# print(dataset)

In [7]:
# train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
# print(train_ds[0])

In [8]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg

        # Get model and dataset details from registry
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

        self.model = None
        self.tokenizer = None
        self.train_ds = None
        self.eval_ds = None
        self.trainer = None

    # -----------------------------
    # Load Model + Apply LoRA
    # -----------------------------
    def load_model(self):
        print("Loading model:", self.model_name)

        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        print("Model loaded and LoRA applied.")
        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):
        print("Loading dataset:", self.dataset_info["name"])

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        # Use small subset for demo/testing
        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(
            format_sample,
            remove_columns=raw.column_names
        )

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        print("Train samples:", len(self.train_ds))
        print("Eval samples:", len(self.eval_ds))

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):
        print("Building trainer...")

        FastVisionModel.for_training(self.model)

        training_args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",

            # Important for vision-language fine-tuning
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},

            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(
                self.model,
                self.tokenizer
            ),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=training_args,
        )

        print("Trainer ready.")
        return self

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self):
        print("Training started for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()
        print("Training completed.")
        return self

    # -----------------------------
    # Save Model
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)

        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)

        print("Model saved to:", self.cfg.save_dir)
        return self

    # -----------------------------
    # Quick Inference Test
    # -----------------------------
    def quick_infer(self, sample_index=0):
        print("Running quick inference...")

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": self.dataset_info["instruction"]},
                    {"type": "image"},
                ],
            }
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(
            self.tokenizer,
            skip_prompt=True
        )

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
            do_sample=True,
        )

        return self

    # -----------------------------
    # Full Pipeline Runner
    # -----------------------------
    def run(self):
        self.load_model()
        self.prepare_data()
        self.build_trainer()
        self.train()
        self.save()
        self.quick_infer()

In [9]:
cfg = VisionFTConfig(
    model_key="qwen2_vl_2b",
    dataset_key="iphone_custom",
)

In [10]:
# # Create config
# cfg = VisionFTConfig()

In [11]:
# trainer = VisionFineTuner(cfg)
# trainer.run()

In [12]:
# Create fine-tuner object
trainer = VisionFineTuner(cfg)

In [13]:
# Step 1: Load model and tokenizer
trainer.load_model()

Loading model: unsloth/Qwen2-VL-2B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.6.8: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Model loaded and LoRA applied.


In [14]:
# Step 2: Prepare training and evaluation dataset
trainer.prepare_data()

Loading dataset: sunny199/iphone5_vlm


README.md:   0%|          | 0.00/610 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.68k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.45k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Train samples: 3
Eval samples: 1


In [15]:
# Step 3: Build SFT trainer
trainer.build_trainer()

Building trainer...
Unsloth: Model does not have a default image size - using 512
Trainer ready.


In [17]:
# Step 4: Train model
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 2 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Training started for 2 epochs


Step,Training Loss


Training completed.


In [18]:
# Step 5: Save trained LoRA model and tokenizer
trainer.save()

Model saved to: vlm_lora_output


In [19]:
# Step 6: Test model with one sample image
trainer.quick_infer()

Running quick inference...
The image shows an iPhone product with a white background and a black frame.<|im_end|>
